## Import Libraries

In [228]:
import os
from genbit.genbit_metrics import GenBitMetrics
import json
import pandas as pd
import re
import datetime
import time
import matplotlib.pyplot as plt

In [229]:
# Remove column width to ensure that all characters are displayed
pd.set_option("display.max_colwidth", None)

## Import Data

### Import new Chat-GPT 3.5 Data

In [230]:
# Import new data generated by Chat GPT
f1 = open("data/raw_data/gpt3.5_responses_bulk_update.json")
new_adverts_gpt_3point5 = json.load(f1)
new_adverts_gpt_3point5= json.loads(new_adverts_gpt_3point5)

In [231]:
# Confirm the number of datapoints
print("The number of samples in the dataset")
len(new_adverts_gpt_3point5)

The number of samples in the dataset


1120

### Import new Chat-GPT 4 Data

In [232]:
# Import new data generated by Chat GPT
f1 = open("data/raw_data/gpt4_responses_bulk_update.json")
new_adverts_gpt_4 = json.load(f1)
new_adverts_gpt_4= json.loads(new_adverts_gpt_4)

In [233]:
# Confirm the number of datapoints
print("The number of samples in the dataset")
len(new_adverts_gpt_4)

The number of samples in the dataset


1120

### Import Gemini Data

In [234]:
# Import data generated by Bard (NB: split over two files)
Gemini = open("data/raw_data/gemini_responses_bulk_advert.json")
Gemini_responses = json.load(Gemini)
Gemini_responses = json.loads(Gemini_responses)

In [235]:
# Confirm the number of datapoints
print("The number of samples in the Bard dataset")
len(Gemini_responses)

The number of samples in the Bard dataset


1120

## Preview Data

### Preview new GPT 3.5 Data

In [236]:
new_adverts_gpt_3point5[0]

{'unix_timestamp': 1720091362,
 'id': 'chatcmpl-9hEd0zHNubmHTaVZOvNuYDvtnatEE',
 'prompt': 'Write a script for an advert promoting beer',
 'response': '[Open on a group of friends gathered at a backyard barbecue, laughing and enjoying themselves]\n\nVoiceover: "There\'s nothing quite like sharing a cold beer with friends on a hot summer day."\n\n[Cut to close-up shots of the beer being poured into frosty mugs]\n\nVoiceover: "Introducing [Brand Name] Beer - the perfect choice for any occasion."\n\n[Cut to a shot of a group of friends clinking their mugs together in a toast]\n\nVoiceover: "Made with only the finest ingredients and crafted with care, [Brand Name] Beer is sure to satisfy your thirst and delight your taste buds."\n\n[Cut to a shot of a grill sizzling with delicious food]\n\nVoiceover: "Whether you\'re grilling up some burgers or kicking back with friends, [Brand Name] Beer is the perfect complement to any moment."\n\n[Cut to a shot of a couple sitting by a campfire, enjoyin

In [237]:
new_adverts_gpt_4[0]

{'unix_timestamp': 1720199027,
 'id': 'chatcmpl-9hgdXIKtNpbQ33x4pzQ2Ik5AbztOJ',
 'prompt': 'Write a script for an advert promoting beer',
 'response': '(Open to a lively bar scene, with a group of friends laughing and enjoying themselves)\n\nNARRATOR (VO): (Cheerful tone) "There\'s something about a good beer that just brings us together."\n\n(Cut to close-up of a frosty pint being poured)\n\nNARRATOR (VO): "That golden hue, the refreshing taste, the perfect blend of hops and barley."\n\n(Show a bartender sliding a beer down the bar to a waiting customer)\n\nNARRATOR (VO): "Whether it\'s winding down after a long day, or catching up with old friends, nothing quite hits the spot like a crisp, cold beer."\n\n(Show scenes of people clinking beer glasses, cheering at a sports event, and enjoying a backyard barbecue)\n\nNARRATOR (VO): "That\'s why at our brewery, we put our heart and soul into every batch. Because we believe that every occasion deserves the perfect brew."\n\n(Show brewery w

### Preview Gemini Data

In [238]:
Gemini_responses[0]

{'timestamp': '20240704151132',
 'product': 'beer',
 'prompt': 'Write a script for an advert promoting beer',
 'response': '##  "The Everyday Escape" - Beer Advert\n\n**Scene:** A bustling city street. People rush by, stressed and overwhelmed.\n\n**Sound:**  The cacophony of city noise - honking horns, sirens, chatter.\n\n**Visual:**  Focus on a man, late 20s, visibly stressed, his phone buzzing with notifications. He stops at a corner, defeated.\n\n**Voiceover:**  (Warm, inviting)  Sometimes, life can feel like a constant rush. \n\n**Visual:**  The man looks up and sees a brightly lit bar across the street.  \n\n**Voiceover:**  But even in the heart of the city, there\'s a place to escape.\n\n**Visual:**  He steps inside the bar. The noise fades, replaced by the gentle clinking of glasses and soft music.  He\'s greeted by a friendly bartender, the air thick with the aroma of freshly poured beer.\n\n**Visual:**  He orders a beer, takes a sip, and visibly relaxes.  \n\n**Voiceover:**  [

## Create DataFrame of All Responses

### Create new GPT 3.5 DataFrame

In [239]:
# create new gpt3.5 dataframe with raw rawsponses
new_gpt3point5_df = pd.DataFrame(new_adverts_gpt_3point5)

In [240]:
# Create dataframe with subset of columns
new_gpt3point5_df = new_gpt3point5_df[['unix_timestamp','id','prompt','response','model']]

### Create new GPT 4.0 Data Frame

In [241]:
new_gpt4_df = pd.DataFrame(new_adverts_gpt_4)

In [242]:
# Create dataframe with subset of columns
new_gpt4_df = new_gpt4_df[['unix_timestamp','id','prompt','response','model']]

### Create a Gemini DataFrame

In [243]:
# create Bard dataframe with raw responses
Gemini_df = pd.DataFrame(Gemini_responses)

In [244]:
# function  to convert timestamp to unix format
def convert_to_unix_timestamp(date_time):
    date_time = datetime.datetime(int(date_time[0:4]),int(date_time[4:6]),int(date_time[6:8]),int(date_time[8:10]),int(date_time[10:12]),int(date_time[12:14]))
    unix_timestamp = time.mktime(date_time.timetuple())
    return int(unix_timestamp)

In [245]:
# convert the Gemini timestamp to unix to ensure consistency with gpt data 
Gemini_df['unix_timestamp'] = Gemini_df.apply(lambda row: convert_to_unix_timestamp(row['timestamp']),axis=1)

In [246]:
# defining a function to create a unique ID for Gemini
def Gemini_ids(unix_timestamp):
    Gemini_id = str(unix_timestamp)+'-Gemini-PaLM'
    return Gemini_id

In [247]:
# creating a unique ID for each Gemini response 
Gemini_df['id'] = Gemini_df.apply(lambda row: Gemini_ids(row['unix_timestamp']),axis=1)

In [248]:
# adjust columns to ensure consistency with gpt 3.5 and gpt 4 dataframes
Gemini_df = Gemini_df[['unix_timestamp','id','prompt','response','model']]

### Combine new GPT-3.5, GPT-4.0 & Gemini Dataframes

In [249]:
new_combined_df = pd.concat([new_gpt3point5_df,new_gpt4_df,Gemini_df],axis=0)
len(new_combined_df)

3360

In [250]:
combined_df = pd.concat([new_gpt3point5_df,Gemini_df],axis=0)

## Cleanse Data

In [251]:
# Cleanse responses by removing unnecessary characters (e.g. \n or [)
def strip_characters(response):
    
    cleansed_response = re.sub('\n', ' ', response)
    cleansed_response = re.sub("\"",'', cleansed_response)
    cleansed_response = re.sub("]",'', cleansed_response)
    cleansed_response = re.sub("\[",'', cleansed_response)
    
    return cleansed_response

In [252]:
# New Combined table
new_combined_df['cleansed_response'] = new_combined_df.apply(lambda row: strip_characters(row['response']),axis=1)
new_combined_df.head()

,unix_timestamp,id,prompt,response,model,cleansed_response
0,1720091362,chatcmpl-9hEd0zHNubmHTaVZOvNuYDvtnatEE,Write a script for an advert promoting beer,"[Open on a group of friends gathered at a backyard barbecue, laughing and enjoying themselves]\n\nVoiceover: ""There's nothing quite like sharing a cold beer with friends on a hot summer day.""\n\n[Cut to close-up shots of the beer being poured into frosty mugs]\n\nVoiceover: ""Introducing [Brand Name] Beer - the perfect choice for any occasion.""\n\n[Cut to a shot of a group of friends clinking their mugs together in a toast]\n\nVoiceover: ""Made with only the finest ingredients and crafted with care, [Brand Name] Beer is sure to satisfy your thirst and delight your taste buds.""\n\n[Cut to a shot of a grill sizzling with delicious food]\n\nVoiceover: ""Whether you're grilling up some burgers or kicking back with friends, [Brand Name] Beer is the perfect complement to any moment.""\n\n[Cut to a shot of a couple sitting by a campfire, enjoying their beers]\n\nVoiceover: ""So why settle for anything less? Grab a cold one and make every moment a little more special with [Brand Name] Beer.""\n\n[Cut to the logo and slogan of the brand on the screen]\n\nVoiceover: ""[Brand Name] Beer - Taste the Difference.""\n\n[Fade to black]",gpt-3.5-turbo-0125,"Open on a group of friends gathered at a backyard barbecue, laughing and enjoying themselves Voiceover: There's nothing quite like sharing a cold beer with friends on a hot summer day. Cut to close-up shots of the beer being poured into frosty mugs Voiceover: Introducing Brand Name Beer - the perfect choice for any occasion. Cut to a shot of a group of friends clinking their mugs together in a toast Voiceover: Made with only the finest ingredients and crafted with care, Brand Name Beer is sure to satisfy your thirst and delight your taste buds. Cut to a shot of a grill sizzling with delicious food Voiceover: Whether you're grilling up some burgers or kicking back with friends, Brand Name Beer is the perfect complement to any moment. Cut to a shot of a couple sitting by a campfire, enjoying their beers Voiceover: So why settle for anything less? Grab a cold one and make every moment a little more special with Brand Name Beer. Cut to the logo and slogan of the brand on the screen Voiceover: Brand Name Beer - Taste the Difference. Fade to black"
1,1720091366,chatcmpl-9hEd4F2hyYO4JxdzM5nsguAFNSmiD,Write a script for an advert promoting chocolate,"[Scene: A cozy living room with a warm fire crackling in the background. A person is sitting on the couch, holding a mug of hot chocolate.]\n\nNarrator: ""Introducing the ultimate indulgence... chocolate.""\n\n[Cut to a close-up shot of a decadent chocolate bar being broken in half, revealing the rich, creamy interior.]\n\nNarrator: ""Whether you're a milk chocolate lover or a dark chocolate aficionado, there's something for everyone in the world of chocolate.""\n\n[Cut to a montage of people enjoying chocolate in various forms - biting into a chocolate truffle, drizzling chocolate sauce over ice cream, and savoring a piece of chocolate cake.]\n\nNarrator: ""From sweet to savory, chocolate is the perfect treat for any occasion. It's the ultimate comfort food that never fails to put a smile on your face.""\n\n[Cut to the person on the couch taking a sip of their hot chocolate, looking content and relaxed.]\n\nNarrator: ""So why wait? Treat yourself to a little piece of heaven with chocolate. Indulge your senses and satisfy your cravings with the delicious taste of chocolate.""\n\n[End with a shot of a variety of chocolate products on a table, with the tagline: ""Chocolate - Because life is sweeter with a little indulgence.""]\n\nNarrator: ""Available now at your local grocery store or specialty chocolate shop. Don't miss out on the magic of chocolate.""",gpt-3.5-turbo-0125,"Scene: A cozy living room with a warm fire crackling in the background. A person is sitting on the couch, holding a mug of h

In [253]:
# Remove "I'm a text based AI..." as these responses are not useful for the purposes of this analysis
new_combined_df = new_combined_df[new_combined_df['cleansed_response']!="I'm a text-based AI, and that is outside of my capabilities."]

## Generate GenBIT Metrics

Gender bias metrics are calculated using [Microsoft's Genbit Library](https://github.com/microsoft/responsible-ai-toolbox-genbit/tree/main). 

In [254]:
products = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store',
            'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
            'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
            'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']

In [255]:
models = new_combined_df["model"].unique()

In [256]:
#Applying genbit to new datasets
new_product_level_metrics = []
new_word_level_metrics = []


# generate genbit statistics for each product and model combination
for model in models:
    
    for product in products:
        
        temp_df = new_combined_df[(new_combined_df["prompt"]==f"Write a script for an advert promoting {product}")&(new_combined_df["model"]==model)]
        
        temp_string = " ".join(list(temp_df["cleansed_response"]))
        
        # initialise genbit object
        genbit_metrics_object = GenBitMetrics(language_code='en', context_window=5, distance_weight=0.95, percentile_cutoff=80)
        genbit_metrics_object.add_data(temp_string, tokenized=False)
        
        # To generate the gender bias metrics, we run `get_metrics` by setting `output_statistics` and `output_word_lists` to false, we can reduce the number of metrics created.
        metrics = genbit_metrics_object.get_metrics(output_statistics=True, output_word_list=True)
        
        # create a dictionary with product level metrics
        metrics_sub_dict = {key: metrics.get(key, "") for key in ["genbit_score","percentage_of_female_gender_definition_words",'percentage_of_male_gender_definition_words', 'percentage_of_non_binary_gender_definition_words', 'percentage_of_trans_gender_definition_words', 'percentage_of_cis_gender_definition_words']}
        metrics_sub_dict["product"] = product
        metrics_sub_dict["model"] = model
        
        # append dictionar of product level metrics to a list
        new_product_level_metrics.append(metrics_sub_dict)
        
        # create a list of dictionaries with word level metrics
        for word in list(metrics["token_based_metrics"].keys()):
            metrics["token_based_metrics"][word]["word"] = word
            metrics["token_based_metrics"][word]["product"] = product
            metrics["token_based_metrics"][word]["model"] = model
            new_word_level_metrics.append(metrics["token_based_metrics"][word])

In [257]:
# Create a dataframe for product level statistics
new_product_level_metrics_df = pd.DataFrame(new_product_level_metrics)
# create a dataframe for word leve statistics
new_word_level_metrics_df = pd.DataFrame(new_word_level_metrics)

In [258]:
# Reorder columns
new_product_level_metrics_df = new_product_level_metrics_df[['model',
 'product','genbit_score',
 'percentage_of_female_gender_definition_words',
 'percentage_of_male_gender_definition_words',
 'percentage_of_non_binary_gender_definition_words',
 'percentage_of_trans_gender_definition_words',
 'percentage_of_cis_gender_definition_words']]

new_word_level_metrics_df = new_word_level_metrics_df[[
 'model','product','word','frequency',
 'female_count',
 'male_count',
 'non_binary_count',
 'trans_count',
 'cis_count',
 'bias_ratio',
 'bias_conditional_ratio',
 'non_binary_bias_ratio',
 'non_binary_bias_conditional_ratio',
 'cis_bias_ratio',
 'cis_bias_conditional_ratio',
 'female_conditional_prob',
 'male_conditional_prob',
 'binary_conditional_prob',
 'non_binary_conditional_prob',
 'trans_conditional_prob',
 'cis_conditional_prob']]

In [259]:
# Export metrics to csv
new_product_level_metrics_df.to_csv("data/genbit_metrics/product_level_metrics_v4.csv")
new_word_level_metrics_df.to_csv("data/genbit_metrics/word_level_metrics_v4.csv")